<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.9em; font-weight:bold; margin:0.6em 0 0.3em 0;">מחברת 3.2: סיווג, האם המטופל יגיע לליקוי קל?</p>

<p dir="rtl" style="direction:rtl; text-align:right;">זהו המשך מודרך של חלק ב במפגש 3. נאמן שני מודלי סיווג על אותם מטופלים, ננבא את מטופלי הבדיקה, ונשווה ביניהם. השאלה הקלינית: <b>האם המטופל יגיע בשלב הכרוני לליקוי קל (<span dir="ltr">FMA-UE</span> מעל 42)?</b> המטרה היא הקטגוריה <code dir="ltr">outcome_mild</code>, ו"אירוע" הוא <code dir="ltr">mild</code>. משך מוערך: כ-20 דקות. בכל תא פקודה אחת שמדפיסה תוצאה אחת.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>כך עובד כל תא בדיקה:</b> אם התשובה נכונה יודפס <code dir="ltr">OK</code>. אם לא, תופיע שגיאה, וזה בסדר: תקנו והריצו שוב. כל מקום שבו כתוב <code dir="ltr">___</code> מחכה להשלמה שלכם.</p>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">הכנת הסביבה</p>

<p dir="rtl" style="direction:rtl; text-align:right;">התא הבא מתקין רק את החבילות הדרושות (<code dir="ltr">tidymodels</code> ו-<code dir="ltr">kknn</code>) ממאגר בינארי מהיר, ומדפיס כמה זמן ההתקנה נמשכה. הריצו אותו פעם אחת בתחילת המפגש.</p>

In [ ]:
# Install only the packages the course needs, from a binary repository (fast)
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
        paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
# tidymodels + kknn: about 100 packages in all, 43 more than tidyverse alone. From the Posit binary
# repository there is no compiling, but the time on a fresh Colab session was NOT measured here
# (a rough guess is a few minutes). Run this cell first, before class starts.
needed  <- c("tidymodels", "kknn")   # add packages here when a notebook needs them
missing <- setdiff(needed, rownames(installed.packages()))   # skip installed ones
t <- system.time(if (length(missing) > 0) install.packages(missing))
cat("Installed:", if (length(missing)) paste(missing, collapse = ", ") else "nothing new",
    "| elapsed seconds:", round(t[["elapsed"]], 1), "\n")
suppressPackageStartupMessages(library(tidymodels))   # recipes, parsnip, rsample, yardstick, tune, ggplot2, dplyr ...
tidymodels_prefer()                                   # when two packages share a name, prefer the tidymodels one

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">הכנה מוכנה מראש (ממחברת 3.1)</p>

<p dir="rtl" style="direction:rtl; text-align:right;">שלושת התאים הבאים חוזרים על מה שעשינו במחברת 3.1: טעינה, פיצול (אותו <code dir="ltr">set.seed(820)</code>) ומתכון. הריצו אותם בלי לשנות. המתכון לומד מהאימון בלבד, ולכן <code dir="ltr">test_baked</code> מוכנה להערכה סופית.</p>

In [ ]:
source("https://raw.githubusercontent.com/ShayOfir/DSAI4PT_Zfat_Students/main/R/load_course_data.R")

In [ ]:
stroke <- load_course_data("stroke_clean") |>        # the cleaned table: one row per patient
  as_tibble() |>                                      # make it a tibble
  mutate(sex = factor(sex, levels = c("F", "M")),     # categories are stored as factors
         lesion_type = factor(lesion_type, levels = c("ischemic", "hemorrhagic", "mixed")))

stroke_ml <- stroke |>
  select(id, fma_sum_chr,                                   # id and the chronic score (the target)
         fma_sum_sub, fma_a_sub, fma_bc_sub,                # features known at the subacute visit ...
         age, sex, lesion_type, lesion_vol, tao_sub) |>     # ... as in chapter 8 of the book
  mutate(outcome_mild = factor(if_else(fma_sum_chr >= 43, "mild", "not_mild"),   # the categorical target
                               levels = c("mild", "not_mild")))

In [ ]:
set.seed(820)                                                 # fixed seed: everyone gets the same split
stroke_split <- initial_split(stroke_ml, prop = 0.75, strata = outcome_mild)   # 75% training, 25% test
stroke_train <- training(stroke_split)                        # the training table
stroke_test  <- testing(stroke_split)                         # the test table (locked until the end)

In [ ]:
stroke_rec <- recipe(outcome_mild ~ fma_sum_sub + age + sex + lesion_vol,
                     data = stroke_train) |>      # the plan: which columns, and which is the target
  step_impute_mode(sex) |>                         # fill a missing category with the most common one
  step_log(lesion_vol) |>                          # log of the lesion volume (it is very skewed)
  step_normalize(all_numeric_predictors()) |>      # z-score for every numeric predictor
  step_dummy(all_nominal_predictors())             # categories -> 0/1 columns

In [ ]:
stroke_rec_prep <- prep(stroke_rec)                           # learn the numbers from the training table
train_baked <- bake(stroke_rec_prep, new_data = stroke_train)   # apply to the training table
test_baked  <- bake(stroke_rec_prep, new_data = stroke_test)    # apply to the test table (same numbers)

In [ ]:
dim(train_baked)   # rows and columns of the prepared training table: 36 patients, 4 predictors and the target

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 1: K-NN, שואלים את המטופלים הדומים ביותר</p>

<p dir="rtl" style="direction:rtl; text-align:right;">כדי לסווג מטופל חדש, <b>K-NN</b> (K השכנים הקרובים) מוצא את K המטופלים הדומים לו ביותר באימון (המרחק מחושב על המאפיינים המתוקננים) ומצביע לפי רובם. זה אותו רעיון של "מטופלים דומים" ממפגש 2. ב-<code dir="ltr">tidymodels</code> מגדירים מודל בשלושה חלקים: <b>סוג</b> המודל (<code dir="ltr">nearest_neighbor()</code> עם מספר השכנים), <b>מצב</b> (<code dir="ltr">set_mode()</code>: סיווג או רגרסיה), ואז מאמנים בעזרת <code dir="ltr">fit()</code>. דוגמה: המפרט.</p>

In [ ]:
knn_spec <- nearest_neighbor(neighbors = 5, weight_func = "rectangular") |>   # K = 5, every neighbour has an equal vote
  set_mode("classification")                                                      # predict a category
knn_spec

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> <code dir="ltr">fit()</code> מקבלת את המפרט, נוסחה ואת טבלת האימון. הנוסחה <code dir="ltr">outcome_mild ~ .</code> אומרת: המטרה היא <code dir="ltr">outcome_mild</code>, והמאפיינים הם "כל שאר העמודות". אמנו את המודל על <code dir="ltr">train_baked</code>.</p>

In [ ]:
knn_fit <- fit(knn_spec, outcome_mild ~ ., data = ___)   # TODO: which table is used for training?

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(inherits(knn_fit, "model_fit"), nrow(predict(knn_fit, new_data = train_baked)) == 36)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> ניבוי: <code dir="ltr">predict()</code> מחזירה את הקטגוריה המנובאת (<code dir="ltr">.pred_class</code>). דוגמה על חמשת המטופלים הראשונים בבדיקה:</p>

In [ ]:
predict(knn_fit, new_data = test_baked) |> head(5)   # predicted category of the first 5 test patients

<p dir="rtl" style="direction:rtl; text-align:right;">כדי להעריך, צריך את התשובה האמיתית ואת הניבוי באותה טבלה. עם <code dir="ltr">type = "prob"</code> מקבלים גם את ההסתברות לכל קטגוריה (<code dir="ltr">.pred_mild</code>, <code dir="ltr">.pred_not_mild</code>). <code dir="ltr">bind_cols()</code> מדביקה טבלאות זו לצד זו. זה התא שנשתמש בו שוב ושוב:</p>

In [ ]:
knn_results <- test_baked |>
  select(outcome_mild) |>                                          # the true category
  bind_cols(predict(knn_fit, new_data = test_baked),               # predicted category
            predict(knn_fit, new_data = test_baked, type = "prob"))   # predicted probabilities
knn_results |> head(4)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(knn_results) == 13, all(c("outcome_mild", ".pred_class", ".pred_mild") %in% names(knn_results)))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 2: רגרסיה לוגיסטית, מודל שמחשב הסתברות</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>רגרסיה לוגיסטית</b> מחשבת לכל מטופל הסתברות לליקוי קל, ומסווגת "קל" אם ההסתברות מעל 0.5. בניגוד ל-K-NN, היא לומדת מספרים (מקדמים) מהאימון. המבנה ב-<code dir="ltr">tidymodels</code> זהה: מפרט, <code dir="ltr">fit()</code>, <code dir="ltr">predict()</code>. הסוג הוא <code dir="ltr">logistic_reg()</code> (המצב הוא תמיד סיווג).</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> הגדירו את המפרט <code dir="ltr">logit_spec</code> ואמנו אותו, באותו אופן כמו ב-K-NN.</p>

In [ ]:
logit_spec <- ___()                                                  # TODO: the model type
logit_fit  <- fit(logit_spec, outcome_mild ~ ., data = ___)           # TODO: which table?

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(inherits(logit_fit, "model_fit"), inherits(logit_fit$fit, "glm"))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> בנו את <code dir="ltr">logit_results</code> כמו <code dir="ltr">knn_results</code>: האמת, הקטגוריה המנובאת וההסתברויות, מהמודל הלוגיסטי ועל טבלת הבדיקה.</p>

In [ ]:
logit_results <- test_baked |>
  select(outcome_mild) |>                                           # the true category
  bind_cols(predict(___, new_data = test_baked),                    # TODO: the model to use
            predict(___, new_data = test_baked, type = "prob"))     # TODO: the same model
logit_results |> head(4)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(logit_results) == 13, all(c("outcome_mild", ".pred_class", ".pred_mild") %in% names(logit_results)))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 3: מטריצת הבלבול, תא אחרי תא</p>

<p dir="rtl" style="direction:rtl; text-align:right;">מטריצת בלבול סופרת את המטופלים לפי <b>מה המודל ניבא</b> (שורות) ו<b>מה קרה באמת</b> (עמודות). ארבעה תאים: אמיתי חיובי (<span dir="ltr">TP</span>: ניבא "קל" והיה קל), שלילי כוזב (<span dir="ltr">FN</span>: ניבא "לא קל" אבל היה קל), חיובי כוזב (<span dir="ltr">FP</span>) ושלילי אמיתי (<span dir="ltr">TN</span>). דוגמה ל-K-NN:</p>

In [ ]:
knn_cm <- conf_mat(knn_results, truth = outcome_mild, estimate = .pred_class)   # count patients per cell
knn_cm

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> עשו אותו דבר למודל הלוגיסטי.</p>

In [ ]:
logit_cm <- conf_mat(logit_results, truth = ___, estimate = ___)   # TODO: the true column and the predicted column
logit_cm

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(sum(logit_cm$table) == 13)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 4: רגישות, סגוליות ודיוק במילים קליניות</p>

<ul dir="rtl" style="direction:rtl; text-align:right;">
<li><b>רגישות (sensitivity):</b> מבין מי שבאמת יגיע לליקוי קל, כמה המודל זיהה? <code dir="ltr">TP / (TP + FN)</code></li>
<li><b>סגוליות (specificity):</b> מבין מי שבאמת לא יגיע לליקוי קל, כמה המודל זיהה? <code dir="ltr">TN / (TN + FP)</code></li>
<li><b>דיוק (accuracy):</b> כמה מכל המטופלים סווגו נכון? <code dir="ltr">(TP + TN) / הכול</code></li>
</ul>

<p dir="rtl" style="direction:rtl; text-align:right;">אפשר לשלוף את הספירות מהמטריצה: <code dir="ltr">knn_cm&#36;table["mild", "mild"]</code> = <span dir="ltr">TP</span> (שורה = ניבוי, עמודה = אמת). דוגמה לחישוב ידני של הרגישות של K-NN:</p>

In [ ]:
knn_tp <- knn_cm$table["mild", "mild"]           # predicted mild, truly mild
knn_fn <- knn_cm$table["not_mild", "mild"]       # predicted not mild, truly mild
knn_sens_manual <- knn_tp / (knn_tp + knn_fn)    # sensitivity by hand
knn_sens_manual

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> חשבו ביד את הסגוליות של המודל הלוגיסטי: מבין מי שבאמת "לא קל", כמה זוהו? <code dir="ltr">TN</code> = ניבוי "לא קל" ואמת "לא קל"; <code dir="ltr">FP</code> = ניבוי "קל" ואמת "לא קל".</p>

In [ ]:
logit_tn <- logit_cm$table["not_mild", "not_mild"]     # predicted not mild, truly not mild
logit_fp <- logit_cm$table["___", "not_mild"]          # TODO: the predicted category of a false positive
logit_spec_manual <- logit_tn / (logit_tn + logit_fp)  # specificity by hand
logit_spec_manual

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(abs(logit_spec_manual - spec(logit_results, truth = outcome_mild, estimate = .pred_class)$.estimate) < 1e-9)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;">לא חייבים לחשב ביד: <code dir="ltr">metric_set()</code> בונה פונקציה שמחשבת כמה מדדים בבת אחת. דוגמה ל-K-NN:</p>

In [ ]:
cls_metrics <- metric_set(accuracy, sens, spec)               # a function that computes three metrics
cls_metrics(knn_results, truth = outcome_mild, estimate = .pred_class)   # K-NN on the test set

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> חשבו את אותם שלושה מדדים למודל הלוגיסטי ושמרו ב-<code dir="ltr">logit_metrics</code>.</p>

In [ ]:
logit_metrics <- cls_metrics(___, truth = outcome_mild, estimate = .pred_class)   # TODO: the results table of the logistic model
logit_metrics

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(logit_metrics) == 3, all(c("accuracy", "sens", "spec") %in% logit_metrics$.metric))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 5: משווים, ובודקים מול ניחוש פשוט</p>

<p dir="rtl" style="direction:rtl; text-align:right;">שני המודלים בטבלה אחת. <code dir="ltr">bind_rows(.id = ...)</code> מדביקה טבלאות זו מתחת לזו ורושמת בעמודה חדשה מאיזו טבלה כל שורה, ו-<code dir="ltr">pivot_wider()</code> הופכת את המדדים לעמודות:</p>

In [ ]:
compare_cls <- bind_rows(knn = cls_metrics(knn_results, truth = outcome_mild, estimate = .pred_class),
                         logistic = logit_metrics, .id = "model") |>   # one table, a column for the model
  select(model, .metric, .estimate) |>
  pivot_wider(names_from = .metric, values_from = .estimate)           # one row per model
compare_cls

<p dir="rtl" style="direction:rtl; text-align:right;">ומה היה קורה עם "מודל" טיפש שמנבא <b>כל</b> מטופל כ"קל"? הדיוק שלו הוא פשוט שיעור ה"קל" בקבוצת הבדיקה. חשבו אותו.</p>

In [ ]:
baseline_accuracy <- mean(test_baked$outcome_mild == ___)   # TODO: the category the silly model always predicts
baseline_accuracy

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(abs(baseline_accuracy - mean(stroke_test$outcome_mild == "mild")) < 1e-9)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>תובנה.</b> האם הדיוק של המודלים טוב יותר מהניחוש "כולם קל"? ואילו מדד מגלה כמה המודל מזהה את המטופלים שלא יגיעו לליקוי קל? (זכרו: בבדיקה יש 13 מטופלים בלבד, ומטופל אחד שווה כ-8 נקודות אחוז.)</p>
<p dir="rtl" style="direction:rtl; text-align:right;">כתבו כאן משפט או שניים: ...</p>